# 예제 02. 고객 데이터 분석과 정기예금 가입 패턴 탐색
### 고객관리 실무와 연결되는 핵심 예제 · 1일 차 · 2시간

> **실행 방법**: 위에서부터 순서대로 실행하세요 (`Shift + Enter`).

| 항목 | 내용 |
|---|---|
| 데이터 | UCI Bank Marketing (`bank-full`) · 포르투갈 은행의 전화 마케팅 기록 45,211건 (익명화) |
| 출처 | UCI Machine Learning Repository (Moro, Cortez & Rita, 2014) · CC BY 4.0 |
| 오늘의 결과물 | 고객군별 정기예금 가입률 분석표 (`segment_rates.csv`) |

**오늘 해결할 질문**: "어떤 고객이 정기예금에 더 많이 가입했을까?"

> ⚠️ **데이터 사용 전 약속**
> - 포르투갈 은행의 **2008~2010년** 마케팅 기록입니다. 한국 저축은행의 현재 고객정보가 아닙니다.
> - 알 수 있는 것은 **가입 여부(yes/no)** 뿐입니다. 예금액 증가나 고객 이탈은 측정하지 않습니다.
> - 연령·직업 등은 **집단 단위 통계 교육**에만 씁니다. 개인 고객을 자동으로 영업 대상으로 고르는 근거로 쓰지 않습니다.

## 0. 실습 준비

아래 셀을 실행하면 실습 데이터를 **GitHub에서 자동으로 내려받습니다.** (강의장 인터넷이 막혀 **[파일 선택]** 버튼이 나오면, 강사가 배포한 파일을 고르면 됩니다.)
`데이터 준비 완료: True`가 나오면 성공입니다.

In [ ]:
import os
import urllib.request
import pandas as pd
import plotly.express as px

DATA_URL = "https://raw.githubusercontent.com/jetsonai/AIFinancialDataAnalysis/main/Day1/data/"
FILE = "bank_customers.csv"
if not os.path.exists(FILE):          # 파일이 없으면 GitHub에서 내려받습니다
    try:
        urllib.request.urlretrieve(DATA_URL + FILE, FILE)
    except Exception:                 # 인터넷이 막혔을 때만 직접 올리기
        from google.colab import files
        files.upload()
print("데이터 준비 완료:", os.path.exists(FILE))

## 02-1. 고객 데이터 읽기

예제 01과 똑같이 `pd.read_csv()`로 읽습니다. 이번에는 한 행이 **고객 한 명**입니다.

In [ ]:
df = pd.read_csv(FILE)
print("고객 수:", df.shape[0], "명 / 항목 수:", df.shape[1], "개")
df.head()

## 02-2. 컬럼 설명표

| 컬럼 | 뜻 | 값 예시 |
|---|---|---|
| `age` | 나이 | 18 ~ 95 |
| `job` | 직업 | admin.(사무직), blue-collar(생산·현장직), management(관리직) … |
| `marital` | 결혼 상태 | married(기혼), single(미혼), divorced(이혼·사별) |
| `education` | 최종 학력 | primary(초등), secondary(중등), tertiary(대학 이상), unknown |
| `default` | 채무불이행(신용불량) 이력 | yes / no |
| `balance` | 연평균 계좌 잔액 (**유로**) | 마이너스도 있음 |
| `housing` | 주택담보대출 보유 | yes / no |
| `loan` | 개인대출 보유 | yes / no |
| `contact` | 연락 방법 | cellular(휴대폰), telephone(유선), unknown |
| `day`, `month` | 마지막 연락 날짜(일), 월 | 5, may |
| `duration` | 마지막 통화 시간(초) | 통화가 **끝나야** 알 수 있는 값 |
| `campaign` | 이번 캠페인에서 연락한 횟수 | 1, 2, … |
| `pdays` | 이전 캠페인 뒤 지난 일수 | **-1 = 이전 연락 없음** |
| `previous` | 이전 캠페인 연락 횟수 | 0, 1, … |
| `poutcome` | 이전 캠페인 결과 | success(가입), failure(미가입), other, unknown(이전 연락 없음 등) |
| **`y`** | **정기예금 가입 여부 (분석 목표)** | **yes / no** |

직업 이름을 한국어로 바꾼 `직업` 항목을 하나 더 만들어 둡니다. `map()`은 사전(dict)대로 값을 바꿔 줍니다.

In [ ]:
직업명 = {
    "admin.": "사무직", "blue-collar": "생산·현장직", "entrepreneur": "사업가",
    "housemaid": "가사도우미", "management": "관리직", "retired": "은퇴자",
    "self-employed": "자영업", "services": "서비스직", "student": "학생",
    "technician": "기술직", "unemployed": "무직", "unknown": "미상",
}
df["직업"] = df["job"].map(직업명)
df[["job", "직업"]].head()

## 02-3. 자료형과 빈 값 확인

In [ ]:
df.info()

In [ ]:
print("비어 있는 칸:", df.isna().sum().sum(), "개")

# 빈칸은 없지만, 'unknown'(모름)이라고 적힌 칸이 있습니다
(df == "unknown").sum()[lambda s: s > 0]

> 💡 **핵심 정리** — 실무 데이터의 "모름"은 빈칸 대신 `unknown`, `-1`, `999` 같은 **약속된 값**으로 들어오는 경우가 많습니다.
> `isna()`가 0이라고 안심하지 말고 값의 뜻을 확인합니다. (`poutcome`의 unknown 대부분은 "이전 캠페인 연락 없음"입니다.)

## 02-4. 고객 특성 파악하기 — 연령 · 직업 · 잔액 · 대출

In [ ]:
df[["age", "balance"]].describe().round(1)

In [ ]:
df["직업"].value_counts()

In [ ]:
print("[개인대출 loan]")
print(df["loan"].value_counts(), "\n")
print("[주택담보대출 housing]")
print(df["housing"].value_counts())

## 02-5. 정기예금 가입 여부 (y)

`(df["y"] == "yes")`는 고객마다 가입이면 True(1), 아니면 False(0)를 만듭니다.
그 **평균**이 곧 **가입률**입니다. 이 한 줄이 오늘 분석의 핵심입니다.

In [ ]:
print(df["y"].value_counts())

전체가입률 = (df["y"] == "yes").mean() * 100
print("\n전체 정기예금 가입률:", round(전체가입률, 1), "%")

> 💡 **핵심 정리** — 전체 가입률은 약 **11.7%**, 100명에게 연락하면 약 12명이 가입했습니다.
> 이제부터 고객군별 가입률을 이 **11.7%와 비교**하며 읽습니다.

## 02-6. 직업별 정기예금 가입률

과정 구성안에 실린 코드 그대로입니다.
`groupby("job")`으로 직업별로 묶고, 각 묶음에서 yes 비율을 계산해 높은 순으로 정렬합니다.

In [ ]:
# 직업별 정기예금 가입률(%)
result = (
    df.groupby("job")["y"]
    .apply(lambda x: (x == "yes").mean() * 100)
    .round(1)
    .sort_values(ascending=False)
)
display(result)

## 02-7. 가입률표 만드는 함수 준비

가입률만 보면 **고객이 몇 명인 집단인지** 알 수 없습니다. 고객 수가 적은 집단은 가입률이 크게 흔들립니다.
그래서 **고객수 · 가입자수 · 가입률**을 한 번에 보여 주는 함수를 준비했습니다.
앞으로는 `가입률표("항목")`만 바꿔 쓰면 됩니다. (함수 안 코드는 지금 다 이해하지 않아도 됩니다.)

In [ ]:
def 가입률표(기준):
    표 = df.groupby(기준, observed=True)["y"].agg(
        고객수="size",
        가입자수=lambda x: (x == "yes").sum(),
    )
    표["가입률(%)"] = (표["가입자수"] / 표["고객수"] * 100).round(1)
    return 표

가입률표("직업").sort_values("가입률(%)", ascending=False)

> 💡 학생·은퇴자의 가입률이 높지만, 학생은 **938명**으로 다른 직업보다 훨씬 적습니다. 가입률과 고객수를 **함께** 읽습니다.

## 02-8. 연령대별 가입률

나이를 그대로 쓰면 값이 너무 많으니 `pd.cut()`으로 **구간**을 나눕니다.
`bins`는 경계값, `labels`는 구간 이름입니다. (0, 29] → 20대 이하, (29, 39] → 30대 …

In [ ]:
df["연령대"] = pd.cut(df["age"],
                     bins=[0, 29, 39, 49, 59, 100],
                     labels=["20대 이하", "30대", "40대", "50대", "60대 이상"])

가입률표("연령대")

## 02-9. 잔액 구간별 가입률

잔액(유로)도 구간으로 나눕니다. `float("inf")`는 "끝없이 큰 수(무한대)"라는 뜻입니다.

In [ ]:
무한 = float("inf")
df["잔액구간"] = pd.cut(df["balance"],
                     bins=[-무한, -1, 0, 1000, 5000, 무한],
                     labels=["마이너스", "0유로", "1~1,000", "1,001~5,000", "5,000 초과"])

가입률표("잔액구간")

## 02-10. 대출 보유 여부에 따른 가입률

In [ ]:
가입률표("loan")        # 개인대출

In [ ]:
가입률표("housing")     # 주택담보대출

## 02-11. 과거 캠페인 반응에 따른 가입률

이전 캠페인에서 가입했던(success) 고객은 이번에 어떻게 반응했을까요?

In [ ]:
가입률표("poutcome").sort_values("가입률(%)", ascending=False)

> 💡 **핵심 정리** — 이전 캠페인에 가입했던 고객의 가입률은 **64.7%**로, 전체 평균(11.7%)의 5배가 넘습니다.
> 다만 이런 고객은 1,511명(3.3%)뿐입니다. "**반응률이 높은 집단**"과 "**규모가 큰 집단**"은 다를 수 있습니다.

## 02-12. 🔧 조건 바꿔 보기 (실습)

`기준`을 바꾸고 실행해 보세요: `"직업"`, `"연령대"`, `"잔액구간"`, `"marital"`, `"education"`, `"loan"`, `"housing"`, `"contact"`, `"poutcome"`

**[선택]** 두 항목을 함께 보려면 대괄호로 묶습니다. 예: `["연령대", "loan"]`

In [ ]:
기준 = "education"       # ← 바꿔 보세요
가입률표(기준)

## 02-13. 고객군별 가입률 분석표 완성 & 저장 ⭐ 결과물

지금까지 만든 표를 `pd.concat()`으로 **하나의 분석표**로 합칩니다.
예제 03(대시보드)에서 이어 쓸 수 있도록 두 파일을 저장합니다.

- `segment_rates.csv` : 고객군별 가입률 분석표
- `bank_customers_prepared.csv` : `직업` · `연령대` · `잔액구간`을 추가한 고객 데이터

In [ ]:
분석표 = pd.concat({
    "직업": 가입률표("직업"),
    "연령대": 가입률표("연령대"),
    "잔액구간": 가입률표("잔액구간"),
    "개인대출": 가입률표("loan"),
    "주택담보대출": 가입률표("housing"),
    "과거캠페인": 가입률표("poutcome"),
}, names=["고객군 기준", "구분"])

분석표

In [ ]:
분석표.to_csv("segment_rates.csv", encoding="utf-8-sig")
df.to_csv("bank_customers_prepared.csv", index=False, encoding="utf-8-sig")
print("저장 완료: segment_rates.csv, bank_customers_prepared.csv")

**[선택]** 내 컴퓨터로 내려받기 (Colab에서만 동작, Excel에서 바로 열 수 있습니다)

In [ ]:
from google.colab import files
files.download("segment_rates.csv")

## 02-14. [선택] 대시보드 맛보기

분석표 하나를 막대그래프로 그려 봅니다. 예제 03에서 이런 그래프들을 한 화면에 모읍니다.

In [ ]:
연령표 = 가입률표("연령대").reset_index()
fig = px.bar(연령표, x="연령대", y="가입률(%)", text="가입률(%)",
             hover_data=["고객수"], title="연령대별 정기예금 가입률")
fig.add_hline(y=전체가입률, line_dash="dot", annotation_text="전체 평균")
fig.show()

## 02-15. 해석할 때 꼭 기억할 점 ⚠️

- **함께 나타난 것 ≠ 원인.** "60대 이상의 가입률이 높다"는 사실이 "나이가 들면 가입한다"는 원인을 증명하지 않습니다.
- **고객수가 적은 집단**의 가입률은 크게 흔들립니다. 항상 고객수를 같이 봅니다.
- `duration`(통화 시간)은 통화가 **끝난 뒤에야** 알 수 있으므로, "누구에게 연락할지" 정하는 분석에는 쓰지 않습니다.
- 2008~2010년 포르투갈의 금리·경제 상황에서 나온 결과입니다. 우리 회사 고객에게 그대로 옮기지 않습니다.
- 집단 통계는 **상품·상담 방식 개선 아이디어**를 얻는 데 쓰고, 개인을 자동으로 고르거나 배제하는 근거로 쓰지 않습니다.

**💬 토의**
1. 가입률이 가장 높은 집단과 고객 수가 가장 많은 집단은 같습니까? 영업 자원을 어디에 쓰겠습니까?
2. 개인대출·주택담보대출 보유 고객의 가입률이 낮은 이유로 무엇을 생각해 볼 수 있을까요? 확인하려면 어떤 데이터가 더 필요할까요?

## ✅ 오늘 배운 것

| 하고 싶은 일 | 코드 |
|---|---|
| 값 바꾸기 (영문 → 한글) | `df["job"].map(사전)` |
| 전체 가입률 | `(df["y"] == "yes").mean() * 100` |
| 집단별로 묶어 계산 | `df.groupby("기준")` |
| 숫자를 구간으로 나누기 | `pd.cut(값, bins=[...], labels=[...])` |
| 고객군별 가입률 표 | `가입률표("기준")` |
| 표 여러 개 합치기 · 저장 | `pd.concat({...})`, `.to_csv("파일")` |

## 📝 복습 미션

> 아래 셀에서 **`# ←` 표시가 있는 줄만** 바꿔 실행하세요. 막히면 맨 아래 **정답 셀**을 실행해 결과를 확인합니다.

**미션 1.** 결혼 상태(`marital`)별 가입률표를 만들어 보세요.
**이렇게 나오면 성공!** divorced 11.9% · married 10.1% · single 14.9%

**미션 2.** 학력(`education`)별 가입률표를 만들고, 가입률이 높은 순으로 정렬해 보세요.
**이렇게 나오면 성공!** tertiary 15.0% → unknown 13.6% → secondary 10.6% → primary 8.6%

**다음 예제 전에 이것만은 꼭!**
- [ ] 가입률 = `(y == "yes")`의 평균
- [ ] 가입률은 **고객수와 함께** 읽는다
- [ ] `bank_customers_prepared.csv`가 저장되었는지 왼쪽 📁 파일 탭에서 확인

In [ ]:
# 미션 1
가입률표("직업")          # ← "직업"을 "marital"로

In [ ]:
# 미션 2
가입률표("직업").sort_values("가입률(%)", ascending=False)   # ← "직업"을 "education"으로

### 정답 (먼저 위 셀을 바꿔 실행해 본 뒤 확인하세요)

In [ ]:
# 미션 1 정답
가입률표("marital")

In [ ]:
# 미션 2 정답
가입률표("education").sort_values("가입률(%)", ascending=False)

---
### ▶ 다음 예제 03. 고객 세분화 및 예금 유치 대시보드
오늘 만든 분석표를 **한 화면의 인터랙티브 대시보드**로 만듭니다. 직업군·고객군을 고르면 그래프가 바뀌도록 해 봅니다.